In [2]:
pip install sentence-transformers

  Using cached sentence_transformers-5.7.0-py3-none-any.whl.metadata (18 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-macosx_11_0_arm64.whl.metadata (9.8 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached typer-0.27.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.24.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmat

In [3]:
pip install langchain_chroma

  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached chromadb-1.5.9-cp39-abi3-macosx_11_0_arm64.whl.metadata (5.0 kB)
  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached opentelemetry_api-1.44.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.44.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.44.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-7.1.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-macosx_10_12_universal2.whl.metadata (10 kB)
  Using cached kubernetes-36.0.3-py2.py3-none-any.whl.metadata (1.8 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached jsonpatch-1.33-py2.py3-none-a

In [4]:
pip install langchain-huggingface

  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
Using cached langchain_huggingface-1.2.2-py3-none-any.whl (31 kB)
Note: you may need to restart the kernel to use updated packages.


In [9]:
from pathlib import Path

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma 

DATA_DIR = Path("../data/runbooks")
PERSIST_DIR = Path("../chroma_sentinel")
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

In [13]:
loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
)
raw_docs = loader.load()

print(f"Loaded {len(raw_docs)} documents")
for d in raw_docs:
    print(" -", d.metadata["source"])

100%|██████████| 6/6 [00:00<00:00, 1304.88it/s]

Loaded 6 documents
 - ../data/runbooks/disk-full-logging-volume.md
 - ../data/runbooks/high-cpu-usage-checkout-service.md
 - ../data/runbooks/ssl-certificate-expiration.md
 - ../data/runbooks/connection-pool-exhaustion.md
 - ../data/runbooks/redis-failover-incident.md
 - ../data/runbooks/memory-leak-notification-service.md


In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap=200,
    separators=["\n## ", "\n### ", "\n\n", "\n", " ", ""],
)
chunks = splitter.split_documents(raw_docs)

print(f"{len(raw_docs)} documents -> {len(chunks)} chunks")
print("\n--- sample chunk ---")
print(chunks[0].page_content[:400])
print("metadata:", chunks[0].metadata)

6 documents -> 18 chunks

--- sample chunk ---
---
title: Disk Full on Logging Volume
service: logging-service
severity: critical
tags: [disk, logging, storage, outage]
last_updated: 2026-02-21
---

# Disk Full — Logging Volume

## Symptoms
- Application pods across multiple services begin failing to write logs; some services (those with synchronous log writes) start throwing write errors and become unresponsive entirely.
- `df -h` on the logg
metadata: {'source': '../data/runbooks/disk-full-logging-volume.md'}


In [17]:
embeddings = HuggingFaceEmbeddings(model_name= EMBEDDING_MODEL)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(PERSIST_DIR),
)

print(f"Persisted {len(chunks)} chunks to {PERSIST_DIR.resolve()}")

/Users/priyakumari/Documents/AI_Practise/AI_Projects/Sentinel-Autonomous-SRE-Incident-Response-Copilot/backend/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2212.90it/s]


Persisted 18 chunks to /Users/priyakumari/Documents/AI_Practise/AI_Projects/Sentinel-Autonomous-SRE-Incident-Response-Copilot/chroma_sentinel


In [19]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

test_queries = [
    "What do we do when Redis connection pool is exhausted?",
    "How do we diagnose a memory leak in a Node.js service?",
    "SSL certificate expired on the API gateway, what's the fix?",
    "checkout service using too much CPU",
    "database CPU spiked right after a redis failover",
]

for q in test_queries:
    print(f"\n{'='*80}\nQUERY: {q}\n{'='*80}")
    results = retriever.invoke(q)
    for i, doc in enumerate(results, 1):
        source = Path(doc.metadata["source"]).name
        print(f"\n[{i}] source: {source}")
        print(doc.page_content[:250].replace("\n", " ") + "...")


QUERY: What do we do when Redis connection pool is exhausted?

[1] source: connection-pool-exhaustion.md
## Diagnosis 1. Check current pool utilization: `redis-cli -h $REDIS_HOST CLIENT LIST | wc -l` compared against the configured `max_connections` in `checkout-service`'s Redis client config. 2. Confirm no long-running or leaked connections: look for c...

[2] source: connection-pool-exhaustion.md
--- title: Redis Connection Pool Exhaustion service: checkout-service severity: high tags: [redis, connection-pool, timeout, checkout] last_updated: 2026-03-14 ---  # Redis Connection Pool Exhaustion — checkout-service  ## Symptoms - `checkout-servic...

[3] source: connection-pool-exhaustion.md
## Resolution 1. Immediate mitigation: increase `max_connections` on the pool via the `REDIS_POOL_MAX_CONNECTIONS` env var and restart the affected pods to relieve pressure. 2. Root fix: audit all Redis client instantiations in the codebase; ensure e...

QUERY: How do we diagnose a memory leak in a 

In [1]:
import sys
print(sys.executable)

/Users/priyakumari/Documents/AI_Practise/AI_Projects/Sentinel-Autonomous-SRE-Incident-Response-Copilot/backend/.venv/bin/python
